# PYTHIA — ML Classique

Prédiction de la viralité de vidéos YouTube à partir de features pré-publication.

**Dataset :** FPS/TPS gaming FR — 3 557 vidéos après nettoyage  
**Objectif :** Classification binaire — viral / not_viral  
**Features :** données connues avant publication (titre, durée, heure)  
**Modèles :** Random Forest · XGBoost  
**Métrique principale :** Recall viral

## Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score
from xgboost import XGBClassifier
from config import DATA_DIR

## Chargement et préparation des données

In [2]:
df = pd.read_json(DATA_DIR / "dataset_fps_fr_labeled.json")

features = ["title_length", "has_number", "has_punctuation", "upper_word_count", 
            "day_of_week", "hour", "duration_seconds"]

X = df[features]
y = (df["label"] == "viral").astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Entraînement des modèles

In [3]:
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced"),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=42, scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum())
}

results =[]
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results.append({
        "Modèles": name,
        "Accuracy": f"{accuracy_score(y_test, y_pred):.2f}",
        "Recall viral": f"{recall_score(y_test, y_pred):.2f}",
        "F1 viral" : f"{f1_score(y_test, y_pred):.2f}"
    })

pd.DataFrame(results)

,Modèles,Accuracy,Recall viral,F1 viral
0,Random Forest,0.77,0.14,0.24
1,XGBoost,0.69,0.33,0.35


## Optimisation des hyperparamètres

In [ ]:
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'class_weight': ['balanced']
}

param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.05, 0.1, 0.2]
}

grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=5, scoring='recall', n_jobs=-1)
grid_xgb = GridSearchCV(XGBClassifier(random_state=42), param_grid_xgb, cv=5, scoring='recall', n_jobs=-1)

grid_rf.fit(X_train, y_train)
grid_xgb.fit(X_train, y_train)

results_gs = []
for name, grid in [("Random Forest", grid_rf), ("XGBoost", grid_xgb)]:
    y_pred = grid.best_estimator_.predict(X_test)
    results_gs.append({
        "Modèles": name,
        "Meilleurs hyperparamètres": grid.best_params_,
        "Accuracy": f"{accuracy_score(y_test, y_pred):.2f}",
        "Recall viral": f"{recall_score(y_test, y_pred, pos_label=1):.2f}",
        "F1 viral" : f"{f1_score(y_test, y_pred, pos_label=1):.2f}"
    })

pd.DataFrame(results_gs)